In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from typing import Optional
import torch.nn.functional as F
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import LRScheduler
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from dataset import Multi30kDataset
from lr_scheduler import NoamScheduler
from model import Transformer, make_src_mask, make_tgt_mask

from tqdm import tqdm

from nltk.translate.bleu_score import corpus_bleu
from dataset import *
from train import *
import wandb

In [ ]:

load_checkpoint()

In [2]:
language_dataset = Multi30kDataset(split = "train")
processed_dataset = language_dataset.process_data()


dataset_obj = TranslationDataset(processed_dataset)
dataloader = DataLoader(
    dataset_obj,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

Building vocab, please wait...


100%|██████████| 29000/29000 [00:00<00:00, 37516.86it/s]


In [16]:
sos_idx = 2
eos_idx = 3
device = "cpu"

src, tgt = next(iter(dataloader))
sample_src = src[0].unsqueeze(0)
sample_tgt = tgt[0].unsqueeze(0)
src_mask = make_src_mask(sample_src).to(device)


src_vocab = len(language_dataset.de_vocab)
tgt_vocab = len(language_dataset.en_vocab)

transformer = Transformer(src_vocab_size = src_vocab, tgt_vocab_size = tgt_vocab,
                          d_model = 512, N = 6, num_heads = 8, d_ff = 2048,
                          dropout = 0.1)


load_checkpoint("checkpoint.pt", transformer)

64

In [30]:
prediction_idx = greedy_decode(transformer, src, src_mask, 40, sos_idx, eos_idx, "cpu", break_at_eos=False)

In [31]:
print("Source: ")
for i in sample_src[0]:
    print(language_dataset.de_itos[i.item()], end = " ")

print("\nTarget, ground truth: ")
for i in sample_tgt[0]:
    print(language_dataset.en_itos[i.item()], end = " ")

print("\nTarget, prediction: ")
print("\n-----")
for i in prediction_idx[0]:
    print(language_dataset.en_itos[i.item()], end = " ")

Source: 
<sos> eine alte frau in einem langen mantel überquert die straße mit einem karren, während eine andere frau eine zigarette raucht und zusieht. <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> 
Target, ground truth: 
<sos> an old woman in a long coat with a small cart crosses the street while another woman smoking a cigarette watches. <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> 
Target, prediction: 

-----
<sos> a man in a man in a man in a <eos> a <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> <eos> 

In [ ]:
bleu_score = evaluate_bleu(transformer, dataloader, language_dataset.en_itos)
bleu_score